# REBULEX — embeddingmagibu2 ölçümü (Colab)

Magibu'nun yeni modeli `alibayram/embeddingmagibu2-text-v1` ile 4 indeks kurulur ve **v2** ile **dilekçe**
(`dilekce_v1`) testleriyle ölçülür. Hepsi temiz metin, kısa başlık (kaynak + daire):

| Havuz | Yöntem | Karşılaştırma |
|---|---|---|
| 10k | tüm karar | Magibu v1, Cosmos (10k temiz) |
| 10k | recursive 1024/128 | Magibu v1 (10k temiz) |
| 2k | tüm karar | 3 modelin 2k temiz sonuçları |
| 2k | recursive 1024/128 | 3 modelin 2k temiz sonuçları |

Kod GitHub'dan (`metin-temizleme` dalı), veriler Drive'dan gelir.

**Başlamadan önce**
1. `Çalışma zamanı > Çalışma zamanı türünü değiştir` → **A100** (L4 de olur, biraz yavaş).
2. Drive'da `REBULEX/magibu2_veri.zip` olmalı (bilgisayarda `data/magibu2_veri.zip`).
3. Soldaki 🔑 *Secrets* bölümünde `HF_TOKEN` olmalı (notebook erişimi açık). Model erişim onayı istiyor;
   token'ın hesabı onaylı olmalı.

**Akış:** `Çalışma zamanı > Tümünü çalıştır`. 1. hücrede Drive izni istenir, onayla. Son hücre çalışma zamanını
kapatır. Sonuçlar Drive'da `REBULEX/magibu2_sonuc/` altında: `experiments.csv`, `gece.log` ve sorgu sorgu sonuçlar.

## 1. Drive'ı bağla ve GPU'yu kontrol et

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ayarlar, kod ve kütüphaneler
Yeni model `sentence-transformers>=6.1` ve `transformers>=5.19` istiyor; yalnızca bu Colab oturumunda yükseltilir.
Model kartı fp16 yerine bf16 öneriyor.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/REBULEX"
SONUC = f"{DRIVE_DIR}/magibu2_sonuc"
LOCAL_DATA = "/content/data"
REPO = "https://github.com/ahmetege0/rebulex.git"
BRANCH = "metin-temizleme"

os.environ["REBULEX_DATA"] = LOCAL_DATA        # config.py veriyi buradan okur
os.environ["REBULEX_BATCH_TOKENS"] = "65536"   # L4/A100 için
os.environ["REBULEX_DTYPE"] = "bfloat16"

if not os.path.exists("/content/rebulex"):
    !git clone -q -b {BRANCH} {REPO} /content/rebulex
else:
    !cd /content/rebulex && git fetch -q && git reset -q --hard origin/{BRANCH}
%cd /content/rebulex
!git log --oneline -1
!pip install -q -r requirements.txt "sentence-transformers>=6.1" "transformers>=5.19"
import sentence_transformers, transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN yüklendi")

## 3. Verileri hazırla
Zip Colab diskine açılır: iki havuzun orijinal ve temiz kararları, test setleri, `experiments.csv`.
Drive'da önceki bir oturumun `experiments.csv`'si varsa o geri yüklenir; bitmiş ölçümler atlanır.

In [ ]:
import shutil

os.makedirs(SONUC, exist_ok=True)
if not os.path.exists(f"{LOCAL_DATA}/experiments.csv"):   # oturum içinde eklenen satırlar ezilmesin
    !unzip -q -o {DRIVE_DIR}/magibu2_veri.zip -d {LOCAL_DATA}
    if os.path.exists(f"{SONUC}/experiments.csv"):
        shutil.copy(f"{SONUC}/experiments.csv", f"{LOCAL_DATA}/experiments.csv")
        print("Önceki experiments.csv Drive'dan geri yüklendi")
# önceki oturumda bitmiş ölçümler (eval) geri gelsin ki yeniden çalışmasın
for size in ["n10000", "n2000"]:
    done = f"{SONUC}/{size}/temiz/magibu2_text"
    if os.path.exists(done):
        shutil.copytree(done, f"{LOCAL_DATA}/{size}/temiz/magibu2_text", dirs_exist_ok=True)
!find {LOCAL_DATA} -maxdepth 2 | sort

## 4. Model testi (~1 dk)
Model indirilir, bir sorgu ve bir belge vektörlenir. Erişim yoksa burada hata verir.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("alibayram/embeddingmagibu2-text-v1", device="cuda").to(torch.bfloat16)
print("Önekler:", model.prompts["query"], "|", model.prompts["document"])
q = model.encode(["task: search result | query: kira tespit davası"], normalize_embeddings=True)
d = model.encode(["title: Yargıtay 3. Hukuk Dairesi | text: Kira bedelinin tespiti istemiyle açılan dava..."],
                 normalize_embeddings=True)
print("vektör boyu:", q.shape[1], "| benzerlik:", float(q @ d.T))
del model
torch.cuda.empty_cache()

## 5. Denemeler
Önce 10k (tüm karar, recursive 1024/128), sonra 2k (aynı ikisi). Her adım ayrı komut: biri hata verirse sonraki
yine çalışır. Her adımdan sonra sonuçlar Drive'a kopyalanır (vektör veritabanı `qdrant` hariç).
Çıktı `magibu2_sonuc/gece.log`'a da yazılır.

In [ ]:
from datetime import datetime


def save():
    try:
        shutil.copy(f"{LOCAL_DATA}/experiments.csv", f"{SONUC}/experiments.csv")
        for size in ["n10000", "n2000"]:
            src = f"{LOCAL_DATA}/{size}/temiz/magibu2_text"
            if os.path.exists(src):
                shutil.copytree(src, f"{SONUC}/{size}/temiz/magibu2_text",
                                ignore=shutil.ignore_patterns("qdrant"), dirs_exist_ok=True)
    except Exception as e:
        print("Drive'a kopyalanamadı:", e)


steps = [
    "--size 10000 --model magibu2_text",
    "--size 2000 --model magibu2_text --deneme per_decision recursive_c1024_o128",
]
for step in steps:
    print(f"##### {datetime.now():%H:%M} run_clean_experiments.py {step}", flush=True)
    !python -u run_clean_experiments.py {step} 2>&1 | tee -a {SONUC}/gece.log
    save()
print(f"Bitti: {datetime.now():%H:%M}")

## 6. Sonuçlar ve karşılaştırma

In [ ]:
import pandas as pd

try:
    exp = pd.read_csv(f"{LOCAL_DATA}/experiments.csv", encoding="utf-8-sig")
    new = exp[exp.model == "alibayram/embeddingmagibu2-text-v1"]
    old = exp[exp["not"].astype(str).str.startswith("temiz") & exp.deneme.isin(["per_decision", "recursive_c1024_o128",
                                                                               "recursive_c512_o64"])]
    cols = ["karar_sayisi", "model", "deneme", "test_seti", "ilk1_genis_%", "ilk5_genis_%", "ilk10_genis_%",
            "ilk100_genis_%", "embedding_sn"]
    pd.set_option("display.width", 250)
    display(pd.concat([old, new])[cols].sort_values(["karar_sayisi", "test_seti", "deneme", "model"]))
except Exception as e:
    print("Tablo basılamadı:", e)

## 7. Çalışma zamanını kapat
Boşta birim harcanmasın. Sonuçlar Drive'da kalır.

In [ ]:
from google.colab import runtime
print(f"Çalışma zamanı kapatılıyor: {datetime.now():%H:%M}")
runtime.unassign()